In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, ProcessCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier, FTTransformerHead
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
# PandasConverter converts to pandas, setting 'id' as the index.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-04-03 10:07:09.485864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


3.12.6 (main, Sep 30 2024, 02:19:13) [GCC 9.4.0]
pandas 2.3.3
polars 1.39.0
numpy 2.1.3
matplotlib.pyplot
seaborn 0.13.2
scipy 1.15.2
sklearn 1.5.2
tensorflow 2.20.0
xgboost 3.2.0
lightgbm 4.6.0
catboost 1.2.8
mllabs 0.6.4


In [2]:
from sklearn.linear_model import LinearRegression
X_reg = ['PhoneService', 'DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 
        'StreamingTV_Y', 'TechSupport_Y', 'No_Internet', 'DSL_Y', 'MultipleLines_Y', 'SeniorCitizen']

reg_lr = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: LinearRegression(fit_intercept = True).fit(x[X_reg], x['MonthlyCharges'])
)
reg_lgb = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: lgb.LGBMRegressor(verbose=-1).fit(x[X_reg], x['MonthlyCharges'])
)
df_train = df_train.with_columns(
    MonthlyCharges_exp_lr = reg_lr.predict(df_train[X_reg]),
    MonthlyCharges_exp_lgb = reg_lgb.predict(df_train[X_reg])
).with_columns(
    TotalCharges_exp_lr = pl.col('MonthlyCharges_exp_lr') * pl.col('tenure'),
    tenure_exp_lr = pl.col('TotalCharges') / (pl.col('MonthlyCharges_exp_lr') + 1),
    TotalCharges_exp_lgb = pl.col('MonthlyCharges_exp_lgb') * pl.col('tenure'),
    tenure_exp_lgb = pl.col('TotalCharges') / pl.col('MonthlyCharges_exp_lgb')
)
df_test = df_test.with_columns(
    MonthlyCharges_exp_lr = reg_lr.predict(df_test[X_reg]),
    MonthlyCharges_exp_lgb = reg_lgb.predict(df_test[X_reg])
).with_columns(
    TotalCharges_exp_lr = pl.col('MonthlyCharges_exp_lr') * pl.col('tenure'),
    tenure_exp_lr = pl.col('TotalCharges') / (pl.col('MonthlyCharges_exp_lr') + 1),
    TotalCharges_exp_lgb = pl.col('MonthlyCharges_exp_lgb') * pl.col('tenure'),
    tenure_exp_lgb = pl.col('TotalCharges') / pl.col('MonthlyCharges_exp_lgb')
)
X_exp = ['MonthlyCharges_exp_lr', 'MonthlyCharges_exp_lgb', 'TotalCharges_exp_lr', 'TotalCharges_exp_lgb', 'tenure_exp_lr', 'tenure_exp_lgb']
X_exp_lr = ['MonthlyCharges_exp_lr', 'TotalCharges_exp_lr', 'tenure_exp_lr']
X_exp_lgb = ['MonthlyCharges_exp_lgb', 'TotalCharges_exp_lgb', 'tenure_exp_lgb']

In [3]:
# df_train = df_train.to_pandas()
# df_test = df_test.to_pandas()

In [4]:
!rm -rf exp/cv_agg

In [5]:
from mllabs import DefaultLogger, TqdmProgressSession

In [6]:
if os.path.exists('exp/cv_agg'):
    e_agg = Experimenter.load('exp/cv_agg', df_train, logger=DefaultLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    if e_agg.status == 'closed':
        e_agg.reopen_exp()
else:
    e_agg = Experimenter.create(
        df_train, 'exp/cv_agg', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=2, shuffle=True, random_state=1),
        splitter_params={'y': target}, logger=DefaultLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)
    )

e_agg.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_agg.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)

e_agg.add_collector(
    MetricCollector(
        'AUC', Connector(edges=y_edges, role='head'), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_agg.add_collector(
    ProcessCollector(
        'process', Connector(edges=y_edges, role='head'), df_test,
        output_var=slice(-1, None), method='mean'
    )
)
e_agg.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01
    }
)

e_agg.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_agg.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)

## Neural network
e_agg.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 0, 'validation_fraction': 0})

Markdown(
    e_agg.desc_spec()
)

📁 Created directory: exp/cv_agg


## The experimentation using 5-fold StratifiedKFold

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=2, random_state=1, shuffle=True)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 2 |
| **Inner Folds** | 1 |

In [7]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_agg.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_agg.set_node(
    'kbin', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', None)]}, params = {'n_bins': 10, 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_agg.set_node(
    'kbin_1', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])]},
    params = {'n_bins': [4000, 200], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_agg.set_node(
    'kbin_2', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])]},
    params = {'n_bins': [500, 100], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_agg.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_agg.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_agg.set_node(
    'tgt_num', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('si', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_agg.set_node(
    'freq_num', grp='pre', processor=FrequencyEncoder, method = 'transform', edges = {'X': [('si', None)], **y_edges}, params = {'normalize': True}
)
e_agg.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)
e_agg.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin', None), ('kbin_1', None), ('kbin_2', None)]}
)
e_agg.set_node(
    'b2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [(None, X_bin), (None, X_bin2)]}, params={'to':'str'}
)
e_agg.set_node(
    'b2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('b2s', None)]}
)

e_agg.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)

e_agg.set_node(
    'rb_std', grp='pre', processor=RobustScaler, edges = {'X': [('rb', None)]}, params = {}
)

e_agg.set_node(
    'expr4', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'Monthly_per_Total': pl.col('si__MonthlyCharges') / pl.col('si__TotalCharges'),
        'Monthly_Avg_ratio': pl.col('si__MonthlyCharges') / (pl.col('si__TotalCharges') / pl.col('si__tenure')),
        'tenure_sq': pl.col('si__tenure') ** 2,
    }, 'with_columns': False}
)
service = ['DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 'StreamingTV_Y', 'TechSupport_Y']
e_agg.set_node(
    'expr5', grp='pre', processor = ExprProcessor, edges = {'X': [(None, service)]},
    params={'dict_expr': {
        'Service_cnt':  pl.sum_horizontal(service),
    }, 'with_columns': False}
)
e_agg.set_node(
    'expr4_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr4', None)]}
)
e_agg.set_node(
    'exp_std', grp='pre', processor = StandardScaler, edges = {'X': [(None, X_exp)]}
)
e_agg.build(n_jobs=1)

Building 18 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 18 node(s)


In [8]:
from mllabs.processor import CatPairCombiner
from itertools import combinations

TOP_CATS_FOR_NGRAM = ['Contract', 'DSL_Y', 'PaymentMethod', 'OnlineSecurity']

bigram_cols = [(i1, i2) for i1, i2 in combinations(TOP_CATS_FOR_NGRAM, 2)]
trigram_cols = [(i1, i2, i3) for i1, i2, i3 in combinations(TOP_CATS_FOR_NGRAM, 3)]

e_agg.set_node(
    'bigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM)]}, params={'pairs': bigram_cols}
)
e_agg.set_node(
    'trigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM[:4])]}, params={'pairs': trigram_cols}
)
e_agg.set_node(
    'com3', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, ['Contract', 'InternetService', 'PaymentMethod'])]}, 
    params={'pairs': [('Contract', 'InternetService', 'PaymentMethod')]}
)
e_agg.set_node(
    'tgt_bigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('bigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_agg.set_node(
    'tgt_trigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('trigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_agg.set_node(
    'coov_bigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('bigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_agg.set_node(
    'coov_trigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('trigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_agg.set_node(
    'exprd', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_3': ((pl.col('si__TotalCharges') * 1e-3).cast(pl.Int8) % 10).cast(pl.String).cast(pl.Categorical)
    }, 'with_columns': False}
)
e_agg.build(n_jobs=4)

Building 8 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

2026-04-03 10:07:18.700263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:18.700263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:18.700263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:18.701580: I tensorflow/core/platform/cpu_featu

Build complete: 8 node(s)


In [9]:
i = 6
e_agg.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
#e_agg.exp()
#e_agg.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('test', ascending = False)

{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x7fd71fd3d070>}

In [15]:
i = 9
e_agg.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 7, 'n_estimators': 3000, 'learning_rate': 0.05, 'colsample_bytree':0.5}
)
e_agg.exp()
e_agg.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('test', ascending = False)

Experimenting 1 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Exp complete: 1 node(s)


,test,train
lgb_9,0.917407,0.926816


In [11]:
i = 10
e_agg.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 31, 'n_estimators': 2000, 'learning_rate': 0.02, 'colsample_bytree':0.25}
)
e_agg.exp(n_jobs=4)
e_agg.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('test', ascending = False)

Experimenting 3 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

2026-04-03 10:07:25.339721: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:25.339721: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:25.339722: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-03 10:07:25.355650: I tensorflow/core/platform/cpu_featu

Exp complete: 3 node(s)


,test,train
lgb_10,0.917432,0.929947


In [ ]:
i = 7
e_agg.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_agg.exp(n_jobs=2, gpu_id_list = [0])
e_agg.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('test', ascending = False)

In [19]:
e_agg.get_collector('process')._input_cache

{}

In [ ]:
i = 9
e_agg.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'max_depth': 4, 'n_estimators': 2500, 'learning_rate': 0.03, 'colsample_bytree': 0.35}
)
e_agg.exp()
e_agg.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

In [ ]:
i = 5
e_agg.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), 
              ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'colsample_bylevel':0.5, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_agg.exp()
e_agg.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

In [ ]:
i = 6
e_agg.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={
        'X': [(None, X_nom + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), 
              ('tgt_bigram', None), ('tgt_trigram', None), ('b2c', None)]
    }, params={'colsample_bylevel':0.5, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_agg.exp()
e_agg.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

In [ ]:
e_agg.set_node(
    'nn_4', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('expr4_std', None), ('exp_std', None), ('coov', None), ('exprd', None), 
                 ('coov_bigram', None), ('coov_trigram', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 4, 'learning_rate': 0.0001}
)
e_agg.exp()

In [18]:
e_agg.get_collector('process').get_output().to_pandas().set_index(
    pd.Index(df_test['id'].to_numpy(), name='id')
).mean(axis=1).to_csv('data/submission.csv')